In [4]:
import math
import re
from collections import Counter

class RAGUniversidad:
    def __init__(self, documentos):
        self.documentos = documentos
        self.stopwords = {"el", "la", "los", "las", "un", "una", "unos", "unas", "de", "y", "a", "en", "por", "para", "con", "sobre", "del", "al", "lo", "le", "les", "se", "que", "es", "son", "ser", "estar", "tener", "hacer"}
        self.idf_global = {}

    # Preprocesamiento: minúsculas, eliminar puntuación, filtrar stopwords
    def preprocesar(self, texto):
        texto = texto.lower()
        texto = re.sub(r'[^à-üña-z0-9\s]', '', texto)  # quitar puntuación y mantener tildes/ñ
        palabras = texto.split()
        palabras = [p for p in palabras if p not in self.stopwords and len(p) > 2]
        return palabras

    # Calcular TF-IDF para una colección de documentos
    def calcular_tfidf(self, docs_tokenizados):
        # Frecuencia de término en cada documento
        tf = []
        for doc in docs_tokenizados:
            tf.append(Counter(doc))
        # Frecuencia inversa de documento
        idf = {}
        N = len(docs_tokenizados)
        for doc_tf in tf:
            for term in doc_tf:
                idf[term] = idf.get(term, 0) + 1
        for term in idf:
            idf[term] = math.log(N / (1 + idf[term]))

        self.idf_global = idf # Store global IDF

        # Vectores TF-IDF (diccionarios)
        vectores = []
        for doc_tf in tf:
            vec = {term: freq * idf.get(term, 0) for term, freq in doc_tf.items()}
            vectores.append(vec)
        return vectores

    # Similaridad coseno
    def similitud_coseno(self, vec1, vec2):
        interseccion = set(vec1.keys()) & set(vec2.keys())
        numerador = sum(vec1[term] * vec2[term] for term in interseccion)
        norma1 = math.sqrt(sum(v**2 for v in vec1.values()))
        norma2 = math.sqrt(sum(v**2 for v in vec2.values()))
        if norma1 == 0 or norma2 == 0:
            return 0
        return numerador / (norma1 * norma2)

    # Fragmentar documentos en chunks (oraciones)
    def fragmentar(self, texto, tamano_chunk=100):
        oraciones = re.split(r'[.;!?]\s+', texto)
        chunks = []
        actual = ""
        for oracion in oraciones:
            if len(actual) + len(oracion) < tamano_chunk:
                actual += " " + oracion
            else:
                if actual:
                    chunks.append(actual.strip())
                actual = oracion
        if actual:
            chunks.append(actual.strip())
        return chunks

    # Preparar la base de conocimiento con chunks y sus vectores TF-IDF
    def preparar_base(self):
        self.chunks = []
        for doc in self.documentos:
            self.chunks.extend(self.fragmentar(doc))
        self.chunks_tokenizados = [self.preprocesar(chunk) for chunk in self.chunks]
        self.vectores_chunks = self.calcular_tfidf(self.chunks_tokenizados)

    # Recuperar el chunk más relevante
    def recuperar(self, pregunta):
        if not hasattr(self, 'vectores_chunks'):
            self.preparar_base()
        pregunta_token = self.preprocesar(pregunta)

        # Crear vector TF-IDF para la pregunta usando el idf global de los chunks
        tf_pregunta = Counter(pregunta_token)
        vec_preg = {term: freq * self.idf_global.get(term, 0) for term, freq in tf_pregunta.items()}

        mejor_chunk = None
        mejor_score = -1
        for idx, vec_chunk in enumerate(self.vectores_chunks):
            score = self.similitud_coseno(vec_preg, vec_chunk)
            if score > mejor_score:
                mejor_score = score
                mejor_chunk = self.chunks[idx]
        return mejor_chunk, mejor_score

    def responder(self, pregunta):
        chunk, score = self.recuperar(pregunta)
        if chunk and score > 0.05:
            return f"📘 [Universidad Nacional] {chunk}\n(Relevancia: {score:.2f})"
        else:
            return "❌ No se encontró información relevante en los documentos."

# --- DOCUMENTOS DE EJEMPLO (Universidad Nacional) ---
documentos = [
    "El calendario académico de la Universidad Nacional de Colombia inicia en febrero y termina en noviembre. Las matrículas se realizan en enero y julio.",
    "La biblioteca central Luis Ángel Arango abre de lunes a viernes de 8:00 a 20:00 horas, y los sábados de 9:00 a 17:00.",
    "El proceso de matrícula para pregrado se realiza en las fechas establecidas por cada sede. Normalmente en enero y julio.",
    "La Universidad Nacional tiene sedes en Bogotá, Medellín, Manizales, Palmira y Leticia. Cada sede ofrece programas académicos diversos.",
    "Bienestar universitario ofrece servicios de salud, deporte, cultura y acompañamiento psicosocial a los estudiantes."
]

# --- USO ---
# The if __name__ == "__main__": block is removed to allow the global `documentos`
# and `RAGUniversidad` class to be accessible for the interactive widgets.
# rag = RAGUniversidad(documentos)
# preguntas = [
#     "¿cuándo es la matrícula?",
#     "horario de la biblioteca",
#     "sedes de la universidad",
#     "qué ofrece bienestar universitario"
# ]
# for p in preguntas:
#     print(f"\nPregunta: {p}")
#     print(rag.responder(p))

### Interactuando con el modelo RAG

Ahora, podemos interactuar con el sistema RAG utilizando widgets para hacer preguntas directamente y ver las respuestas.

In [5]:
from IPython.display import display, HTML
import ipywidgets as widgets

# Instanciar el sistema RAG con los documentos de ejemplo
rag_interactive = RAGUniversidad(documentos)
rag_interactive.preparar_base()

In [6]:
def on_button_click(b):
    with output:
        output.clear_output()
        question = question_input.value
        if question:
            response = rag_interactive.responder(question)
            display(HTML(f"<b>Pregunta:</b> {question}<br><b>Respuesta:</b> {response}"))
        else:
            display(HTML("<i>Por favor, ingresa una pregunta.</i>"))

# Crear widgets de entrada y salida
question_input = widgets.Textarea(
    value='',
    placeholder='Escribe tu pregunta aquí...',
    description='Pregunta:',
    disabled=False,
    layout=widgets.Layout(width='500px', height='80px')
)

ask_button = widgets.Button(description="Obtener Respuesta")
output = widgets.Output()

# Vincular el botón a la función
ask_button.on_click(on_button_click)

# Mostrar los widgets
display(question_input, ask_button, output)

Textarea(value='', description='Pregunta:', layout=Layout(height='80px', width='500px'), placeholder='Escribe …

Button(description='Obtener Respuesta', style=ButtonStyle())

Output()